# Day 17 — Feature engineering: encoding, scaling and derived features

Run each cell with **Shift + Enter**, from the top. We use `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

## 1. Introduction: why feature engineering matters

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-017/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import pandas as pd

# The lesson's five students: raw features
df = pd.DataFrame({"study_hours": [2, 4, 6, 3, 8], "attendance": [55, 70, 90, 65, 95],
                   "prev_score": [40, 55, 85, 50, 92], "assignments_done": [3, 5, 9, 4, 10]})
print(df.to_string(index=False))

 study_hours  attendance  prev_score  assignments_done
           2          55          40                 3
           4          70          55                 5
           6          90          85                 9
           3          65          50                 4
           8          95          92                10


In [ ]:
# Feature creation and binning
df["weekly_study_hours"] = df["study_hours"] * 7
df["attendance_category"] = pd.cut(df["attendance"], bins=[0, 60, 80, 100], labels=["Low", "Medium", "High"], include_lowest=True)
df["score_category"] = pd.cut(df["prev_score"], bins=[0, 50, 70, 100], labels=["Low", "Medium", "High"], include_lowest=True)
df["assignment_completion"] = df["assignments_done"] / 10          # completion ratio, assuming 10 assignments were set
print(df.to_string(index=False))

 study_hours  attendance  prev_score  assignments_done  weekly_study_hours attendance_category score_category  assignment_completion
           2          55          40                 3                  14                 Low            Low                    0.3
           4          70          55                 5                  28              Medium         Medium                    0.5
           6          90          85                 9                  42                High           High                    0.9
           3          65          50                 4                  21              Medium            Low                    0.4
           8          95          92                10                  56                High           High                    1.0


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

# Model A vs Model B on our 127 training students: does engineering features help? (pass/fail, 5-fold CV)
data = pd.read_csv("student_data_150.csv")
train_val, test = train_test_split(data, test_size=0.15, random_state=42, stratify=data["Passed"])
y = train_val["Passed"].astype(int)
X = train_val[["StudyHours", "Attendance", "PreviousScore"]].copy()
X["WeeklyStudyHours"] = X["StudyHours"] * 7
X["AttendanceCategory"] = pd.cut(X["Attendance"], [0, 60, 80, 100], labels=["Low", "Medium", "High"], include_lowest=True).astype(str)
X["ScoreCategory"] = pd.cut(X["PreviousScore"], [0, 50, 70, 100], labels=["Low", "Medium", "High"], include_lowest=True).astype(str)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
def model(numeric, categorical=()):
    prep = ColumnTransformer([("num", StandardScaler(), list(numeric))] +
                             ([("cat", OneHotEncoder(handle_unknown="ignore"), list(categorical))] if categorical else []))
    return make_pipeline(prep, LogisticRegression(max_iter=1000))
raw = ["StudyHours", "Attendance", "PreviousScore"]
for name, num, cat in [("A: raw features", raw, ()),
                       ("A + weekly study hours", raw + ["WeeklyStudyHours"], ()),
                       ("Categories only (binned)", ["StudyHours"], ["AttendanceCategory", "ScoreCategory"]),
                       ("Raw + categories", raw, ["AttendanceCategory", "ScoreCategory"])]:
    s = cross_val_score(model(num, cat), X, y, cv=cv)
    print(f"{name:<26} CV accuracy {s.mean():.3f} +/- {s.std():.3f}")

A: raw features            CV accuracy 0.796 +/- 0.062
A + weekly study hours     CV accuracy 0.788 +/- 0.073
Categories only (binned)   CV accuracy 0.781 +/- 0.089
Raw + categories           CV accuracy 0.796 +/- 0.062


In [ ]:
# Data leakage: a feature that would not exist at prediction time. ExamScore decides pass/fail, so a "feature" built
# from it looks perfect in testing and is useless in practice (we never know it before the exam)
X_leak = X[raw].assign(ScoreChange=train_val["ExamScore"] - train_val["PreviousScore"])
for name, cols, frame in [("Raw features", raw, X), ("Raw + ScoreChange (leaks)", raw + ["ScoreChange"], X_leak)]:
    s = cross_val_score(model(cols), frame, y, cv=cv)
    print(f"{name:<26} CV accuracy {s.mean():.3f}")

Raw features               CV accuracy 0.796
Raw + ScoreChange (leaks)  CV accuracy 0.937
